In [ ]:
import pandas as pd

# Menampilkan seluruh kolom
pd.set_option('display.max_columns', None)

In [ ]:
file_path = "/content/drive/MyDrive/Porto/restoran/resto.xlsx"

menu = pd.read_excel(file_path, sheet_name="menu_items")
orders = pd.read_excel(file_path, sheet_name="order_details")

#Data Understanding

Preview dataset

In [ ]:
menu.head()

,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00


In [ ]:
orders.head()

,order_details_id,order_id,order_date,order_time,item_id
0,1,1,2023-01-01 00:00:00,11:38:36,109.0
1,2,2,2023-01-01 00:00:00,11:57:40,108.0
2,3,2,2023-01-01 00:00:00,11:57:40,124.0
3,4,2,2023-01-01 00:00:00,11:57:40,117.0
4,5,2,2023-01-01 00:00:00,11:57:40,129.0


struktur dataset

In [ ]:
menu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   menu_item_id  32 non-null     int64  
 1   item_name     32 non-null     object 
 2   category      32 non-null     object 
 3   price         32 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.1+ KB


In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12234 entries, 0 to 12233
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_details_id  12234 non-null  int64  
 1   order_id          12234 non-null  int64  
 2   order_date        12234 non-null  object 
 3   order_time        12234 non-null  object 
 4   item_id           12097 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 478.0+ KB


jumlah baris dan kolom

In [ ]:
print(f"Menu Dataset   : {menu.shape}")
print(f"Orders Dataset : {orders.shape}")

Menu Dataset   : (32, 4)
Orders Dataset : (12234, 5)


#Data Quality Check

missing value

In [ ]:
print(menu.isnull().sum())

menu_item_id    0
item_name       0
category        0
price           0
dtype: int64


In [ ]:
print(orders.isnull().sum())

order_details_id      0
order_id              0
order_date            0
order_time            0
item_id             137
dtype: int64


mengecek duplicate data

In [ ]:
print("Duplicate Menu   :", menu.duplicated().sum())

Duplicate Menu   : 0


In [ ]:
print("Duplicate Orders :", orders.duplicated().sum())

Duplicate Orders : 0


missing value pada item_id

In [ ]:
orders[orders["item_id"].isna()]

,order_details_id,order_id,order_date,order_time,item_id
121,122,50,2023-01-01 00:00:00,18:41:01,NaN
297,298,125,2023-02-01 00:00:00,20:31:06,NaN
357,358,147,2023-03-01 00:00:00,14:32:51,NaN
386,387,161,2023-03-01 00:00:00,16:43:46,NaN
469,470,200,2023-03-01 00:00:00,22:24:05,NaN
...,...,...,...,...,...
11716,11717,5149,3/28/23,14:48:50,NaN
11903,11904,5225,3/29/23,17:40:52,NaN
11906,11907,5226,3/29/23,17:43:56,NaN
12021,12022,5281,3/30/23,16:56:04,NaN


menghitung jumlah dan presentase missing value pada item_id

In [ ]:
missing_item = orders["item_id"].isnull().sum()

print(f"Missing item_id : {missing_item}")
print(f"Percentage      : {(missing_item/len(orders))*100:.2f}%")

Missing item_id : 137
Percentage      : 1.12%


#Data Cleaning


*   Menghapus transaksi yang tidak memiliki item_id.
*   karena tidak dapat dihubungkan dengan tabel menu.

In [ ]:
orders = orders.dropna(subset=["item_id"])

cek missing value telah di hapus

In [ ]:
print(orders.isnull().sum())

order_details_id    0
order_id            0
order_date          0
order_time          0
item_id             0
dtype: int64


Mengubah tipe data agar sesuai dengan kebutuhan analisis

In [ ]:
orders["item_id"] = orders["item_id"].astype(int)

orders["order_date"] = pd.to_datetime(orders["order_date"])

orders["order_time"] = pd.to_datetime(
    orders["order_time"],
    format="%H:%M:%S"
)

#Persiapan Data

Menggabungkan data transaksi dengan informasi menu.

In [ ]:
restaurant = pd.merge(
    orders,
    menu,
    left_on="item_id",
    right_on="menu_item_id",
    how="left"
)

Memastikan seluruh data berhasil digabungkan

In [ ]:
restaurant.isnull().sum()

,0
order_details_id,0
order_id,0
order_date,0
order_time,0
item_id,0
menu_item_id,0
item_name,0
category,0
price,0


Menambahkan kolom hour untuk menganalisis jam ramai dan jam sepi.

In [ ]:
restaurant["hour"] = restaurant["order_time"].dt.hour

Menyimpan dataset yang telah dibersihkan untuk digunakan pada tahap visualisasi di Power BI.

In [ ]:
restaurant.to_csv(
    "/content/drive/MyDrive/Porto/restoran/versi yg lebih rapi/restaurant_clean.csv",
    index=False
)